# 04 — Benchmark, Export & GGUF
Merge the LoRA adapter, run the base/DAPT/SFT benchmark comparison, and export to GGUF for local Ollama/llama.cpp inference. Run `02` and `03` first.

**Runtime:** T4 GPU (merge + benchmark need it; GGUF conversion itself is CPU-only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/legal-compliance-slm"

In [ ]:
!git clone https://github.com/Shankar-behera/legal-compliance-slm.git
%cd legal-compliance-slm
!pip install -r requirements.txt -q

## Merge LoRA into the DAPT base weights

In [ ]:
from src.export.merge_lora import merge_lora

dapt_final = f"{DRIVE_ROOT}/checkpoints/dapt/final"
adapter_dir = f"{DRIVE_ROOT}/checkpoints/sft/final_adapter"
merged_dir = f"{DRIVE_ROOT}/merged"

merge_lora(base_model_path=dapt_final, adapter_path=adapter_dir, output_path=merged_dir)

## Benchmark: base vs. DAPT vs. SFT

Runs all three checkpoints against `data/eval/compliance_benchmark.jsonl`. **Expand that file with real held-out examples before trusting these numbers** — the shipped file has 3 seed examples, enough to prove the script works, not enough for a real benchmark.

In [ ]:
!python -m src.eval.benchmark \
    --base_model_path Qwen/Qwen2.5-1.5B-Instruct \
    --dapt_model_path "{dapt_final}" \
    --sft_model_path "{adapter_dir}" \
    --benchmark_path data/eval/compliance_benchmark.jsonl \
    --output_path docs/benchmark_results.md \
    --output_json_path docs/benchmark_results.json

In [ ]:
with open("docs/benchmark_results.md") as f:
    print(f.read())

## Convert to GGUF

Builds `llama.cpp` from source, then converts + quantizes the merged model.

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt -q
!cmake -B llama.cpp/build llama.cpp -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --config Release -j$(nproc)

In [ ]:
from src.export.convert_gguf import convert_to_gguf

gguf_path = convert_to_gguf(
    merged_model_path=merged_dir,
    llama_cpp_dir="llama.cpp",
    output_dir="models/gguf",
    quant_type="Q4_K_M",
)
print("GGUF ready:", gguf_path)

## Copy the GGUF to Drive so you can download it locally

In [ ]:
import shutil, os
os.makedirs(f"{DRIVE_ROOT}/models/gguf", exist_ok=True)
shutil.copy(gguf_path, f"{DRIVE_ROOT}/models/gguf/{os.path.basename(gguf_path)}")
print("Copied to Drive. Download from your Drive UI, then see inference/run_ollama.md")

## Next: serve it

Download the `.gguf` file locally, then follow `inference/run_ollama.md`, or run the FastAPI service directly against the merged (pre-quantization) model:
```bash
MODEL_PATH=<path-to-merged-model> uvicorn inference.app_fastapi:app --port 8000
python inference/app_gradio.py --api_url http://localhost:8000
```
